# Sentiment Analysis with RoBERTa

## Task 1.2 — Transformer-Based Model

In this notebook, we implement a Transformer-based sentiment classifier using RoBERTa.

We train and evaluate the model on three different datasets:
- 1K Amazon Reviews
- 25K Amazon Reviews
- Video Game Reviews

Each dataset is processed and evaluated independently using the same pipeline.
This ensures a fair comparison across models in Task 1.3.

In [1]:
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from pathlib import Path

from transformers import AutoTokenizer, AutoModelForSequenceClassification

from utils import device_check
from utils_BERT import (
    build_bert_loaders,
    fit_bert,
    evaluate_bert,
    plot_confusion_matrix_bert,
)

## Setup

In [2]:
LOG_WANDB = True
NUM_WORKERS = 8
PIN_MEMORY = True
SEED = 1

MODEL_NAME = "roberta-base"
MAX_LENGTH = 128
BATCH_SIZE = 16
NUM_EPOCHS = 5

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = device_check()

SPLITS_DIR = Path("../data/splits")

PyTorch: 2.11.0+cu130 | Python: 3.11.15 | OS: Linux 5.15.0-168-generic
CUDA available: True
GPUs: 1 x NVIDIA GeForce RTX 2080 Ti (11.3 GB)
CUDA: 13.0 | cuDNN: 91900
Using cuda / NVIDIA GeForce RTX 2080 Ti


## Dataset 1 — 1K Amazon Reviews

In [3]:
train_df = pd.read_csv(SPLITS_DIR / "1k_train.csv")
val_df = pd.read_csv(SPLITS_DIR / "1k_val.csv")
test_df = pd.read_csv(SPLITS_DIR / "1k_test.csv")

text_col = "Sentence"
label_col = "Class"

NUM_LABELS = 2
NUM_CLASSES = 2

## Tokenizer and DataLoaders

RoBERTa uses the same Hugging Face pipeline style as BERT, so we tokenize the text and build DataLoaders from the pre-split CSV files.

In [4]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

loaders = build_bert_loaders(
    train_df=train_df,
    val_df=val_df,
    test_df=test_df,
    tokenizer=tokenizer,
    text_col=text_col,
    label_col=label_col,
    max_length=MAX_LENGTH,
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
)

train_loader = loaders["train"]
val_loader = loaders["val"]
test_loader = loaders["test"]

## Model

We fine-tune a pre-trained RoBERTa model for binary sentiment classification.

In [5]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS,
)

model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5)

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
classifier.dense.bias           | MISSING    | 
classifier.out_proj.weight      | MISSING    | 
classifier.dense.weight         | MISSING    | 
classifier.out_proj.bias        | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


## Training

The model is trained on the training split and validated after every epoch.
The best checkpoint is restored automatically based on validation loss.

In [6]:
history_1k = fit_bert(
    model=model,
    optimizer=optimizer,
    criterion=criterion,
    train_loader=train_loader,
    val_loader=val_loader,
    device=device,
    num_epochs=NUM_EPOCHS,
    wandb_kwargs={"project": "roberta-1k", "name": "roberta-1k"},
    log=LOG_WANDB,
)

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.
wandb: Currently logged in as: hamid-sabeti (d7047e-group12) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Epoch 1/1 | train loss 0.2152, train acc 91.71% | val loss 0.2492, val acc 91.69%


Training Accuracy,▁
Training Loss,▁
Validation Accuracy,▁
Validation Loss,▁
Training Accuracy,91.71358
Training Loss,0.21519
Validation Accuracy,91.68889
Validation Loss,0.24917



Restored best weights (val loss 0.2492)


## Final Test Evaluation and Confusion Matrix

The final result is reported on the held-out test set for fair comparison with the other models.

evaluate_bert(model, test_loader, criterion, device, label="RoBERTa-1K")

plot_confusion_matrix_bert(
    model,
    test_loader,
    NUM_CLASSES,
    device,
    ["Negative", "Positive"],
    "RoBERTa 1K Confusion Matrix",
)

## Dataset 2 — 25K Amazon Reviews

In [ ]:
train_df = pd.read_csv(SPLITS_DIR / "25k_train.csv")
val_df = pd.read_csv(SPLITS_DIR / "25k_val.csv")
test_df = pd.read_csv(SPLITS_DIR / "25k_test.csv")

text_col = "Sentence"
label_col = "Class"

NUM_LABELS = 2
NUM_CLASSES = 2

## Tokenizer and DataLoaders

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

loaders = build_bert_loaders(
    train_df=train_df,
    val_df=val_df,
    test_df=test_df,
    tokenizer=tokenizer,
    text_col=text_col,
    label_col=label_col,
    max_length=MAX_LENGTH,
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
)

train_loader = loaders["train"]
val_loader = loaders["val"]
test_loader = loaders["test"]

## Model

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS,
)

model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5)

## Training

In [ ]:
history_25k = fit_bert(
    model=model,
    optimizer=optimizer,
    criterion=criterion,
    train_loader=train_loader,
    val_loader=val_loader,
    device=device,
    num_epochs=NUM_EPOCHS,
    wandb_kwargs={"project": "roberta-25k", "name": "roberta-25k"},
    log=LOG_WANDB,
)

## Final Test Evaluation and Confusion Matrix

In [ ]:
evaluate_bert(model, test_loader, criterion, device, label="RoBERTa-25K")

plot_confusion_matrix_bert(
    model,
    test_loader,
    NUM_CLASSES,
    device,
    ["Negative", "Positive"],
    "RoBERTa 25K Confusion Matrix",
)

## Dataset 3 — Video Game Reviews

In [ ]:
train_df = pd.read_csv(SPLITS_DIR / "vg_train.csv")
val_df = pd.read_csv(SPLITS_DIR / "vg_val.csv")
test_df = pd.read_csv(SPLITS_DIR / "vg_test.csv")

text_col = "Sentence"
label_col = "Class"

# Convert labels from 1-5 → 0-4
train_df[label_col] = train_df[label_col].astype(int) - 1
val_df[label_col] = val_df[label_col].astype(int) - 1
test_df[label_col] = test_df[label_col].astype(int) - 1

NUM_LABELS = 5
NUM_CLASSES = 5

## Tokenizer and DataLoaders

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

loaders = build_bert_loaders(
    train_df=train_df,
    val_df=val_df,
    test_df=test_df,
    tokenizer=tokenizer,
    text_col=text_col,
    label_col=label_col,
    max_length=MAX_LENGTH,
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
)

train_loader = loaders["train"]
val_loader = loaders["val"]
test_loader = loaders["test"]

## Model

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS,
)

model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5)

## Training

In [ ]:
history_vg = fit_bert(
    model=model,
    optimizer=optimizer,
    criterion=criterion,
    train_loader=train_loader,
    val_loader=val_loader,
    device=device,
    num_epochs=NUM_EPOCHS,
    wandb_kwargs={"project": "roberta-vg", "name": "roberta-vg"},
    log=LOG_WANDB,
)

## Final Test Evaluation and Confusion Matrix

In [ ]:
evaluate_bert(model, test_loader, criterion, device, label="RoBERTa-VG")

plot_confusion_matrix_bert(
    model,
    test_loader,
    NUM_CLASSES,
    device,
    ["1", "2", "3", "4", "5"],
    "RoBERTa Confusion Matrix - Video Game Reviews",
)